# Train & evaluate the YOLOv8 helmet detector (Kaggle)

This notebook trains the helmet / bare-head detector used by the monitoring system and
measures it on a held-out test split, producing the **mAP@0.5** and **mAP@0.5:0.95** numbers.

**Before you press Run All:**
1. **Add the dataset:** right panel → *Add Input* → search **"Hard Hat Detection"** (by *andrewmvd*, 5,000 images) → *Add*.
2. **GPU:** *Settings* → *Accelerator* → **GPU T4 x2** (or P100).
3. **Internet:** *Settings* → *Internet* → **On** (needed to install Ultralytics and clone the repo; requires a phone-verified Kaggle account).

Training takes roughly 1–2 hours on a T4. When it finishes, download the files listed in the last cell.

In [ ]:
# Setup: install Ultralytics and download the project code.
import socket, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")

try:
    socket.create_connection(("pypi.org", 443), timeout=5).close()
except OSError:
    raise RuntimeError(
        "No internet. Turn it on: right panel > Settings > Internet = On "
        "(needs a phone-verified account), restart the session, then Run All."
    ) from None

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
if not (REPO_DIR / "training" / "prepare_dataset.py").is_file():
    subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/HarshKnight2477/Helmet-Monitoring-Face-Recognition.git", str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import ultralytics
print("Ready: Ultralytics", ultralytics.__version__, "| project code in", REPO_DIR)

## 1. Find the dataset and convert it to YOLO format (70 / 20 / 10 split, fixed seed)

In [ ]:
from pathlib import Path

matches = [p for p in Path("/kaggle/input").rglob("annotations") if p.is_dir()]
assert matches, "Dataset not found. Add 'Hard Hat Detection' (andrewmvd) via Add Input."
ANNOTATIONS = matches[0]
IMAGES = ANNOTATIONS.parent / "images"
DATASET = Path("/kaggle/working/helmet-yolo")
print("Annotations:", ANNOTATIONS)
print("Images:     ", IMAGES)

In [ ]:
import sys
if "/kaggle/working/repo" not in sys.path:
    sys.path.insert(0, "/kaggle/working/repo")  # in case the session restarted
import json
from training.prepare_dataset import prepare

summary = prepare(ANNOTATIONS, IMAGES, DATASET, classes=("helmet", "head"), val=0.2, test=0.1, seed=42)
print(json.dumps(summary, indent=2))

## 2. Train

`helmet` = head wearing a helmet, `head` = bare head (a violation).
Settings below are a good balance of accuracy and time; `patience` stops early if validation mAP stops improving.

In [ ]:
import torch
from ultralytics import YOLO

BASE_MODEL = "yolov8s.pt"   # try "yolov8m.pt" for higher accuracy (about 2x slower)
EPOCHS = 60
IMAGE_SIZE = 640
BATCH = 16

model = YOLO(BASE_MODEL)
model.train(
    data=str(DATASET / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    patience=20,
    seed=42,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/kaggle/working/runs",
    name="helmet",
    exist_ok=True,
)

## 3. Evaluate on the held-out test split

The test images were never used for training or for choosing the best epoch, so these are the numbers to report.

In [ ]:
import sys
if "/kaggle/working/repo" not in sys.path:
    sys.path.insert(0, "/kaggle/working/repo")  # in case the session restarted
from training.evaluate import evaluate, write_reports

BEST = Path("/kaggle/working/runs/helmet/weights/best.pt")
metrics = evaluate(BEST, DATASET / "data.yaml", split="test", image_size=IMAGE_SIZE)
write_reports(metrics, Path("/kaggle/working/reports"))
print(f"mAP@0.5      = {metrics['map50']:.1%}")
print(f"mAP@0.5:0.95 = {metrics['map50_95']:.1%}")
print(f"Precision    = {metrics['precision']:.1%}")
print(f"Recall       = {metrics['recall']:.1%}")

In [ ]:
from IPython.display import Markdown
Markdown(Path("/kaggle/working/reports/detection_metrics.md").read_text())

## 4. Package the files to download

In [ ]:
import shutil

shutil.copy(BEST, "/kaggle/working/helmet_yolov8.pt")
shutil.copy(DATASET / "split_summary.json", "/kaggle/working/reports/split_summary.json")
shutil.copy("/kaggle/working/runs/helmet/results.csv", "/kaggle/working/reports/training_results.csv")
shutil.make_archive("/kaggle/working/reports", "zip", "/kaggle/working/reports")
shutil.rmtree("/kaggle/working/repo")       # keep the Output tab tidy
shutil.rmtree(DATASET)
print("Done. Download from the Output tab (right panel):")
print("  helmet_yolov8.pt  -> put in the project's models/ folder")
print("  reports.zip       -> unzip into the project's reports/ folder")